### Models

#### Invoking Model

In [ ]:
from langchain.chat_models import init_chat_model
# from dotenv import load_dotenv
# load_dotenv()
import os, sys

# os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model(
    model="ollama:qwen2.5:7b",
    temperature = 0.5,
    max_tokens = 2500,
    max_retries = 2,
    timeout= 30,
)

response = model.invoke("What is a parrot ?")
print(response.content)

#### Streaming

In [ ]:
stream = model.stream("Which is the most powerful animal in entire world")
for chunk in stream:
    print(chunk.text, end="|", flush=True)

#### Batching

##### With Entire Output

In [ ]:
batch_response = model.batch([
    "What is purpose of 2 kidneys in human body?",
    "Why elephants can not fly?",
    "Why mens have beard but women have not ?"
])

for response in batch_response:
    print(response.text)

##### When Completed

In [ ]:
batch_response = model.batch_as_completed([
    "What is purpose of 2 kidneys in human body?",
    "Why elephants can not fly?",
    "Why mens have beard but women have not ?"
])

print(batch_response)

### Tools (Invoking Tools)

In [ ]:
from langchain.tools import tool
import getpass

@tool
def getUserName() -> str:
    "Get windows user name"
    return f"User Name {getpass.getuser()}"
tools = [getUserName]
tools_by_name = {tool.name: tool for tool in tools}
model_with_tools = model.bind_tools(tools)
messages = [{"role":"user", "content":"what this current user name"}]
ai_response = model_with_tools.invoke(messages)
messages.append(ai_response)
for tool_call in ai_response.tool_calls:
    toolInfo = tools_by_name[tool_call["name"]]
    tool_result = toolInfo.invoke(tool_call)
    messages.append(tool_result)
final_response = model_with_tools.invoke(messages)
print(final_response.text)


#### Structured output

In [ ]:
from pydantic import BaseModel, Field

class BotMessage(BaseModel):
    calorie: str = Field(description="Total Amount Of calorie to be taken")
    weight: int = Field(description="Weight of the person in lbs/kg")
    diet_type: str = Field(description="Keto Diet or Normal Diet")
    protein_intake: str = Field(description="Amount of protein according to body weight")

model_with_structure = model.with_structured_output(schema=BotMessage)
model_with_structure.invoke(f"Please preapare a plan for making me fit with in 90 days, if i am a 5.6ft and having 65kg body weight")

#### Strctured Output (included RAW response)

In [ ]:
user_prompt = "Please preapare a plan for making me fit with in 90 days, if i am a 5.6ft and having 65kg body weight"
ms = model.with_structured_output(schema=BotMessage, include_raw=True)
response = ms.invoke(user_prompt)
response

#### Using Multi Modal Agent

In [ ]:
response = model.invoke("Create a picture of a calm monk")
print(response.content_blocks)
# For This We need MutiModal Agents

#### Reasoning

In [ ]:
stream = model.stream("Preapare a Road map for GEN-AI Engineer profile")
for chunk in stream:
    resoning_steps = [r for r in chunk.content_blocks if r["type"] == "reasoning"]
    print(resoning_steps if resoning_steps else chunk.text)

#### Resoning Efforts

In [ ]:
from langchain_ollama import ChatOllama

model = ChatOllama(
    model="qwen2.5:7b",
    temperature=0.5
    # base_url="http://localhost:11434"
)

resp = model.invoke("Where is Area 51 and why its kept secret")
resp.pretty_print()

In [ ]:
resp.pretty_print()

#### Running Server Tool

In [ ]:
server_tool = {"type": "web_search"}
mwt = model.bind_tools([server_tool])
resp = mwt.invoke("What is Cockroach Janta Party ?")
print(resp.content_blocks)

#### Exceptions in Model

In [ ]:
from langchain_core.exceptions import ModelTimeoutError
try:
    response = model.invoke("Nothing !")
    response.pretty_print()
except ModelTimeoutError:
    print("timeout while reaching to model")

#### Rate Limiting

In [64]:
from langchain.rate_limiters import InMemoryRateLimiter

rate_Limit = InMemoryRateLimiter(
    requests_per_second=0.1,
    check_every_n_seconds=0.1,
    max_bucket_size=10
)

model = init_chat_model(
    model="ollama:qwen2.5:7b",
    rate_limiter = rate_Limit
)

resp = model.invoke("What is rate limiting in lanchain models in 50 words")
print(resp.text)
print(resp)
print(resp.usage_metadata)

Rate limiting in Langchain models restricts the number of API requests to prevent overloading the system, ensuring stable performance and reliability, and often necessary for compliance with service usage policies.
content='Rate limiting in Langchain models restricts the number of API requests to prevent overloading the system, ensuring stable performance and reliability, and often necessary for compliance with service usage policies.' additional_kwargs={} response_metadata={'model': 'qwen2.5:7b', 'created_at': '2026-10-03T17:17:35.185884419Z', 'done': True, 'done_reason': 'stop', 'total_duration': 5786247846, 'load_duration': 7342397, 'prompt_eval_count': 43, 'prompt_eval_duration': 268998000, 'eval_count': 37, 'eval_duration': 5501119000, 'logprobs': None, 'model_name': 'qwen2.5:7b', 'model_provider': 'ollama'} id='lc_run--01a102c5-2e49-7e01-bb30-e246a2d4b6cc-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 43, 'output_tokens': 37, 'total_tokens': 80}
{'input_to

#### Callback Handler
`UsageMetadataCallbackHandler`

In [69]:
from langchain_core.callbacks import UsageMetadataCallbackHandler

model1 = init_chat_model(model="ollama:qwen2.5:7b")
model2 = init_chat_model(model="ollama:deepseek-r1:1.5b")
model3 = init_chat_model(model="ollama:gemma3:latest")

callback = UsageMetadataCallbackHandler()
model1.invoke("Do you know bruce lee?", config={"callbacks": [callback]})
model2.invoke("Where is the origin of buddisim ?", config={"callbacks": [callback]})
model3.invoke("Is there any parllel earth?", config={"callbacks": [callback]})
print(callback.usage_metadata)


{'qwen2.5:7b': {'input_tokens': 36, 'output_tokens': 189, 'total_tokens': 225}, 'deepseek-r1:1.5b': {'input_tokens': 12, 'output_tokens': 28, 'total_tokens': 40}, 'gemma3:latest': {'input_tokens': 18, 'output_tokens': 932, 'total_tokens': 950}}


`get_usage_metadata_callback`

In [70]:
from langchain_core.callbacks import get_usage_metadata_callback

model1 = init_chat_model(model="ollama:qwen2.5:7b")
model2 = init_chat_model(model="ollama:deepseek-r1:1.5b")
model3 = init_chat_model(model="ollama:gemma3:latest")

with get_usage_metadata_callback() as cb:
    model1.invoke("Do you know bruce lee?", config={"callbacks": [callback]})
    model2.invoke("Where is the origin of buddisim ?", config={"callbacks": [callback]})
    model3.invoke("Is there any parllel earth?", config={"callbacks": [callback]})
    print(cb.usage_metadata)

{'qwen2.5:7b': {'input_tokens': 36, 'output_tokens': 236, 'total_tokens': 272}, 'deepseek-r1:1.5b': {'input_tokens': 12, 'output_tokens': 28, 'total_tokens': 40}, 'gemma3:latest': {'input_tokens': 18, 'output_tokens': 1087, 'total_tokens': 1105}}


#### Configurable Model

In [ ]:
config_model = init_chat_model(tempreture = 0)
response = config_model.invoke("Who is powerful india or china? Based on armed forces", config={
    "configurable":{
        "model": "ollama:qwen2.5:7b"
    }
})
print(response.text)